# FoodSeg103 Semantic Segmentation with CCNet-Style ResNet-50

This notebook trains a CCNet-style semantic segmentation model with a ResNet-50 backbone on FoodSeg103.

In [ ]:
# Optional installs (uncomment if needed)
# %pip install opencv-python albumentations
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os
import json
import base64
import zlib
import warnings

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm
from torchvision.models import resnet50, ResNet50_Weights

warnings.filterwarnings("ignore")

FORCE_CPU = False
if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

device = torch.device("cpu" if FORCE_CPU else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using device: {device}")

np.random.seed(42)
torch.manual_seed(42)
if device.type == "cuda":
    torch.cuda.manual_seed_all(42)

## 1. Dataset and DataLoader Setup

In [ ]:
DATASET_DIR = "./foodseg103/train"
IMAGES_DIR = os.path.join(DATASET_DIR, "img")
LABELS_DIR = os.path.join(DATASET_DIR, "ann")
META_PATH = os.path.normpath(os.path.join(DATASET_DIR, "..", "meta.json"))

NUM_CLASSES = 104  # 103 foreground + background
TRAIN_IMAGE_SIZE = (512, 512)
BATCH_SIZE = 2
NUM_EPOCHS = 10
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
AUX_LOSS_WEIGHT = 0.2
LABEL_SMOOTHING = 0.05
WARMUP_EPOCHS = 1

MAX_TRAIN_SAMPLES = 64  # set None for full split
MAX_VAL_SAMPLES = 16    # set None for full split

if os.name == "nt":
    NUM_WORKERS = 0
else:
    NUM_WORKERS = min(4, os.cpu_count() or 2)
PIN_MEMORY = device.type == "cuda"

print(f"Dataset dir: {DATASET_DIR}")
print(f"Workers={NUM_WORKERS}, PinMemory={PIN_MEMORY}, Batch={BATCH_SIZE}")
print(f"LR={LEARNING_RATE}, MaxGradNorm={MAX_GRAD_NORM}, AuxWeight={AUX_LOSS_WEIGHT}")

In [ ]:
def load_class_mapping_from_meta(meta_path):
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"meta.json not found: {meta_path}")

    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    classes = meta.get("classes", [])
    titles = []
    seen = set()
    for c in classes:
        title = str(c.get("title", "")).strip()
        if title and title not in seen:
            seen.add(title)
            titles.append(title)

    if not titles:
        raise ValueError(f"No valid class titles in meta file: {meta_path}")

    return {title: idx + 1 for idx, title in enumerate(titles)}


def load_foodseg_mask_from_json(ann_path, image_shape_hw, class_to_idx):
    if not os.path.exists(ann_path):
        raise FileNotFoundError(f"Annotation not found: {ann_path}")

    with open(ann_path, "r", encoding="utf-8") as f:
        ann = json.load(f)

    h, w = int(image_shape_hw[0]), int(image_shape_hw[1])
    mask = np.zeros((h, w), dtype=np.uint8)

    for obj in ann.get("objects", []):
        bmp = obj.get("bitmap")
        if not bmp:
            continue

        data_b64 = bmp.get("data")
        origin = bmp.get("origin", [0, 0])
        if data_b64 is None or len(origin) != 2:
            continue

        raw = zlib.decompress(base64.b64decode(data_b64))
        arr = np.frombuffer(raw, dtype=np.uint8)
        decoded = cv2.imdecode(arr, cv2.IMREAD_UNCHANGED)
        if decoded is None:
            continue

        if decoded.ndim == 3 and decoded.shape[2] == 4:
            alpha = decoded[:, :, 3] > 0
        elif decoded.ndim == 2:
            alpha = decoded > 0
        else:
            alpha = decoded[:, :, 0] > 0

        x0, y0 = int(origin[0]), int(origin[1])
        bh, bw = alpha.shape[:2]
        x1, y1 = x0 + bw, y0 + bh

        if x0 >= w or y0 >= h or x1 <= 0 or y1 <= 0:
            continue

        x0c, y0c = max(0, x0), max(0, y0)
        x1c, y1c = min(w, x1), min(h, y1)
        sx0, sy0 = x0c - x0, y0c - y0
        sx1, sy1 = sx0 + (x1c - x0c), sy0 + (y1c - y0c)

        cls_name = obj.get("classTitle", "unknown")
        cls_idx = class_to_idx.get(cls_name, 0)

        region = alpha[sy0:sy1, sx0:sx1]
        mask[y0c:y1c, x0c:x1c][region] = cls_idx

    return mask


class FoodSeg103Dataset(Dataset):
    def __init__(self, images_dir, labels_dir, image_files, transforms, image_size, class_to_idx):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.image_size = tuple(map(int, image_size))
        self.class_to_idx = dict(class_to_idx)

        self.image_files = []
        self.annotation_paths = {}
        for img_name in image_files:
            ann_path = os.path.join(labels_dir, f"{img_name}.json")
            if os.path.exists(ann_path):
                self.image_files.append(img_name)
                self.annotation_paths[img_name] = ann_path

        if not self.image_files:
            raise RuntimeError("No valid image/annotation pairs found.")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        ann_path = self.annotation_paths[img_name]

        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = load_foodseg_mask_from_json(ann_path, image_shape_hw=image.shape[:2], class_to_idx=self.class_to_idx)

        image = cv2.resize(image, self.image_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, self.image_size, interpolation=cv2.INTER_NEAREST)

        aug = self.transforms(image=image, mask=mask)
        image_t = aug["image"]
        mask_t = aug["mask"]
        if not torch.is_tensor(mask_t):
            mask_t = torch.from_numpy(mask_t)
        return image_t, mask_t.long()


if not (os.path.exists(IMAGES_DIR) and os.path.exists(LABELS_DIR)):
    raise FileNotFoundError("Dataset directories not found. Update DATASET_DIR first.")

class_to_idx = load_class_mapping_from_meta(META_PATH) if os.path.exists(META_PATH) else None
if class_to_idx is None:
    raise FileNotFoundError(f"meta.json missing at {META_PATH}.")

expected_num_classes = len(class_to_idx) + 1
if NUM_CLASSES != expected_num_classes:
    print(f"[Warning] NUM_CLASSES={NUM_CLASSES}, meta implies {expected_num_classes}.")

train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=15, p=0.5),
    A.GaussNoise(p=0.2),
    A.Blur(blur_limit=3, p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
    ToTensorV2(),
])

all_images = sorted([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
split_idx = int(0.8 * len(all_images))
train_images = all_images[:split_idx]
val_images = all_images[split_idx:]

if MAX_TRAIN_SAMPLES is not None:
    train_images = train_images[:MAX_TRAIN_SAMPLES]
if MAX_VAL_SAMPLES is not None:
    val_images = val_images[:MAX_VAL_SAMPLES]

train_dataset = FoodSeg103Dataset(
    images_dir=IMAGES_DIR,
    labels_dir=LABELS_DIR,
    image_files=train_images,
    transforms=train_transforms,
    image_size=TRAIN_IMAGE_SIZE,
    class_to_idx=class_to_idx,
)

val_dataset = FoodSeg103Dataset(
    images_dir=IMAGES_DIR,
    labels_dir=LABELS_DIR,
    image_files=val_images,
    transforms=val_transforms,
    image_size=TRAIN_IMAGE_SIZE,
    class_to_idx=class_to_idx,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print(f"Train images: {len(train_dataset)}")
print(f"Val images: {len(val_dataset)}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 2. Build CCNet-Style Model with ResNet-50

In [ ]:
import math

class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )


def _diag_mask(batch_count, n, device, dtype):
    mask = torch.zeros((batch_count, n, n), device=device, dtype=dtype)
    idx = torch.arange(n, device=device)
    mask[:, idx, idx] = -1e4
    return mask


class CrissCrossAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        inter_channels = max(1, in_channels // 8)
        self.query_conv = nn.Conv2d(in_channels, inter_channels, kernel_size=1)
        self.key_conv = nn.Conv2d(in_channels, inter_channels, kernel_size=1)
        self.value_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.softmax = nn.Softmax(dim=-1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        b, _, h, w = x.shape

        query = self.query_conv(x)
        key = self.key_conv(x)
        value = self.value_conv(x)

        query_H = query.permute(0, 3, 1, 2).contiguous().view(b * w, -1, h).permute(0, 2, 1)
        key_H = key.permute(0, 3, 1, 2).contiguous().view(b * w, -1, h)
        value_H = value.permute(0, 3, 1, 2).contiguous().view(b * w, -1, h)

        query_W = query.permute(0, 2, 1, 3).contiguous().view(b * h, -1, w).permute(0, 2, 1)
        key_W = key.permute(0, 2, 1, 3).contiguous().view(b * h, -1, w)
        value_W = value.permute(0, 2, 1, 3).contiguous().view(b * h, -1, w)

        scale_H = 1.0 / math.sqrt(max(1, query_H.shape[-1]))
        scale_W = 1.0 / math.sqrt(max(1, query_W.shape[-1]))

        energy_H = torch.bmm(query_H, key_H) * scale_H + _diag_mask(b * w, h, x.device, x.dtype)
        energy_H = energy_H.view(b, w, h, h).permute(0, 2, 1, 3)

        energy_W = (torch.bmm(query_W, key_W) * scale_W).view(b, h, w, w)

        attention = self.softmax(torch.cat([energy_H, energy_W], dim=-1))
        att_H = attention[..., :h].permute(0, 2, 1, 3).contiguous().view(b * w, h, h)
        att_W = attention[..., h:].contiguous().view(b * h, w, w)

        out_H = torch.bmm(value_H, att_H.transpose(1, 2)).view(b, w, -1, h).permute(0, 2, 3, 1)
        out_W = torch.bmm(value_W, att_W.transpose(1, 2)).view(b, h, -1, w).permute(0, 2, 1, 3)

        out = self.gamma * (out_H + out_W) + x
        return out


class RCCAHead(nn.Module):
    def __init__(self, in_channels, num_classes, inter_channels=512, recurrence=1):
        super().__init__()
        self.recurrence = recurrence
        self.pre = ConvBNReLU(in_channels, inter_channels, kernel_size=3, padding=1)
        self.cca = CrissCrossAttention(inter_channels)
        self.post = ConvBNReLU(inter_channels, inter_channels, kernel_size=3, padding=1)
        self.classifier = nn.Sequential(
            nn.Dropout2d(0.1),
            nn.Conv2d(inter_channels, num_classes, kernel_size=1),
        )

    def forward(self, x):
        x = self.pre(x)
        for _ in range(self.recurrence):
            x = self.cca(x)
        x = self.post(x)
        return self.classifier(x)


class CCNetResNet50(nn.Module):
    def __init__(self, num_classes, pretrained=True, recurrence=1):
        super().__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        backbone = resnet50(weights=weights, replace_stride_with_dilation=[False, True, True])

        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.head = RCCAHead(in_channels=2048, num_classes=num_classes, inter_channels=512, recurrence=recurrence)
        self.aux_head = nn.Sequential(
            ConvBNReLU(1024, 256, kernel_size=3, padding=1),
            nn.Dropout2d(0.1),
            nn.Conv2d(256, num_classes, kernel_size=1),
        )

    def forward(self, x):
        input_size = x.shape[-2:]

        x = self.stem(x)
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)

        out = self.head(x4)
        out = F.interpolate(out, size=input_size, mode="bilinear", align_corners=False)

        aux = self.aux_head(x3)
        aux = F.interpolate(aux, size=input_size, mode="bilinear", align_corners=False)

        return {"out": out, "aux": aux}


model = CCNetResNet50(num_classes=NUM_CLASSES, pretrained=True, recurrence=1).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Model: CCNet-style ResNet-50")
print(f"Classes: {NUM_CLASSES}")
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

## 3. Training and Validation

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=255, label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.99))


def lr_lambda(epoch):
    warmup_factor = min(1.0, (epoch + 1) / max(1, WARMUP_EPOCHS))
    progress = max(0.0, (epoch + 1 - WARMUP_EPOCHS) / max(1, NUM_EPOCHS - WARMUP_EPOCHS))
    poly_factor = (1.0 - progress) ** 0.9
    return warmup_factor * poly_factor


scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def calculate_iou(pred, target, num_classes):
    ious = []
    for cls in range(1, num_classes):
        pred_cls = (pred == cls).float()
        target_cls = (target == cls).float()

        intersection = (pred_cls * target_cls).sum().item()
        union = (pred_cls + target_cls).sum().item() - intersection
        if union == 0:
            iou = 1.0 if intersection == 0 else 0.0
        else:
            iou = intersection / union
        ious.append(iou)

    return float(np.mean(ious)) if ious else 0.0


def train_one_epoch(model, loader, criterion, optimizer, device, num_classes):
    model.train()
    running_loss = 0.0
    iou_scores = []
    grad_norms = []
    skipped_batches = 0

    with tqdm(loader, desc="Train", leave=False) as pbar:
        for images, masks in pbar:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)

            logits = outputs["out"]
            if not torch.isfinite(logits).all():
                skipped_batches += 1
                continue

            loss = criterion(logits, masks)
            if "aux" in outputs:
                aux_loss = criterion(outputs["aux"], masks)
                loss = loss + AUX_LOSS_WEIGHT * aux_loss

            if not torch.isfinite(loss):
                skipped_batches += 1
                continue

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            if not torch.isfinite(grad_norm):
                optimizer.zero_grad(set_to_none=True)
                skipped_batches += 1
                continue

            optimizer.step()

            with torch.no_grad():
                preds = logits.argmax(dim=1)
                iou_scores.append(calculate_iou(preds, masks, num_classes))
                grad_norms.append(float(grad_norm.item()))

            running_loss += float(loss.item())
            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "g": f"{float(grad_norm):.3f}",
                "skip": skipped_batches,
            }, refresh=True)

    mean_loss = running_loss / max(1, (len(loader) - skipped_batches))
    mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
    mean_grad = float(np.mean(grad_norms)) if grad_norms else 0.0
    return mean_loss, mean_iou, mean_grad, skipped_batches


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, num_classes):
    model.eval()
    running_loss = 0.0
    iou_scores = []
    skipped_batches = 0

    with tqdm(loader, desc="Val", leave=False) as pbar:
        for images, masks in pbar:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            logits = outputs["out"]

            if not torch.isfinite(logits).all():
                skipped_batches += 1
                continue

            loss = criterion(logits, masks)
            if not torch.isfinite(loss):
                skipped_batches += 1
                continue

            preds = logits.argmax(dim=1)
            iou_scores.append(calculate_iou(preds, masks, num_classes))

            running_loss += float(loss.item())
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "skip": skipped_batches}, refresh=True)

    mean_loss = running_loss / max(1, (len(loader) - skipped_batches))
    mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
    return mean_loss, mean_iou, skipped_batches


CHECKPOINT_DIR = "./checkpoints_ccnet"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_val_loss = float("inf")
train_losses, val_losses = [], []
train_ious, val_ious = [], []
train_grad_norms = []

print("Starting CCNet training...")
for epoch in range(NUM_EPOCHS):
    tr_loss, tr_iou, tr_grad, tr_skip = train_one_epoch(
        model, train_loader, criterion, optimizer, device, NUM_CLASSES
    )
    va_loss, va_iou, va_skip = validate_one_epoch(
        model, val_loader, criterion, device, NUM_CLASSES
    )

    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_ious.append(tr_iou)
    val_ious.append(va_iou)
    train_grad_norms.append(tr_grad)

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        best_path = os.path.join(CHECKPOINT_DIR, "best_ccnet_resnet50.pth")
        torch.save(model.state_dict(), best_path)
        print(f"Epoch {epoch+1}: saved best checkpoint -> {best_path}")

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"LR: {current_lr:.2e} | "
        f"Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f} | "
        f"Train mIoU: {tr_iou:.4f} | Val mIoU: {va_iou:.4f} | "
        f"GradNorm: {tr_grad:.3f} | Skip(T/V): {tr_skip}/{va_skip}"
    )

print("Training completed.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(train_losses, marker="o", label="Train Loss")
axes[0].plot(val_losses, marker="s", label="Val Loss")
axes[0].set_title("CCNet Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(train_ious, marker="o", label="Train mIoU")
axes[1].plot(val_ious, marker="s", label="Val mIoU")
axes[1].set_title("CCNet mIoU")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("mIoU")
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(train_grad_norms, marker="^", color="tab:red", label="Train Grad Norm")
axes[2].axhline(MAX_GRAD_NORM, linestyle="--", color="gray", label="Clip Threshold")
axes[2].set_title("Gradient Norm")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Norm")
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Best Val Loss: {min(val_losses):.4f}")
print(f"Best Val mIoU: {max(val_ious):.4f}")

In [ ]:
@torch.no_grad()
def predict_on_image(image_path, model, device, image_size=(512, 512)):
    image = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_rs = cv2.resize(image_rgb, image_size, interpolation=cv2.INTER_LINEAR)

    x = image_rs.astype(np.float32) / 255.0
    x = (x - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
    x = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0).to(device)

    out = model(x)["out"]
    pred = out.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    return image_rs, pred


latest_path = os.path.join(CHECKPOINT_DIR, "ccnet_resnet50_last.pth")
torch.save(model.state_dict(), latest_path)
print(f"Saved latest model: {latest_path}")

config_path = os.path.join(CHECKPOINT_DIR, "ccnet_resnet50_config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": "CCNet-Style",
        "backbone": "ResNet-50",
        "num_classes": NUM_CLASSES,
        "image_size": TRAIN_IMAGE_SIZE,
        "epochs": NUM_EPOCHS
    }, f, indent=2)
print(f"Saved config: {config_path}")

sample_file = val_images[0] if len(val_images) > 0 else train_images[0]
sample_path = os.path.join(IMAGES_DIR, sample_file)
ann_path = os.path.join(LABELS_DIR, f"{sample_file}.json")

img, pred = predict_on_image(sample_path, model, device, image_size=TRAIN_IMAGE_SIZE)
gt = load_foodseg_mask_from_json(ann_path, image_shape_hw=img.shape[:2], class_to_idx=class_to_idx)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img)
axes[0].set_title(f"Image: {sample_file}")
axes[0].axis("off")

axes[1].imshow(gt, cmap="tab20")
axes[1].set_title("Ground Truth")
axes[1].axis("off")

axes[2].imshow(pred, cmap="tab20")
axes[2].set_title("CCNet Prediction")
axes[2].axis("off")

plt.tight_layout()
plt.show()